# Agente Doom: percepcion YOLO + Reinforcement Learning

**Teoria de Aprendizaje de Maquinas (UNAL).** Equipo: David (percepcion) y Joshua (entorno + politica).

Agente que juega Doom (ViZDoom, escenario `defend_the_center`: sala abierta tipo torreta)
combinando:

1. **Percepcion:** un detector **YOLOv8** entrenado con un dataset in-domain auto-etiquetado
   desde el propio motor (labels ground-truth de ViZDoom). Convierte cada frame en un vector
   de features (hay enemigo?, a que lado, que tan cerca, vida, ammo...).
2. **Politica:** un agente **Double DQN (Dueling + PER + n-step)** que aprende, sobre ese
   vector, que accion tomar para sobrevivir y matar enemigos que aparecen por todos lados.

```
ViZDoom (defend_the_center) --frame--> YOLO --detecciones--> features (13 dims)
        --> FrameStack (x3 = 39) --> DQN --accion--> ViZDoom
```

Este notebook es **autocontenido**: entrena el detector y el agente de cero y muestra el
codigo de cada modulo inline. Requiere **GPU** (Settings -> Accelerator -> GPU) e Internet ON.

## 1. Setup
Instala ViZDoom (headless), Ultralytics (YOLO) y OpenCV.

In [ ]:
import os
os.environ["DOOM_HEADLESS"] = "1"  # sin pantalla (Kaggle/Colab): ViZDoom usa software renderer
!apt-get -qq install -y libsdl2-2.0-0 > /dev/null 2>&1 || true
!pip -q install vizdoom ultralytics opencv-python-headless
import torch
print("CUDA disponible:", torch.cuda.is_available())


## 2. Escenario

Escribimos el `.cfg` de `defend_the_center` con el layout de 8 botones que usa nuestra
politica. El `.wad` ya viene con el paquete `vizdoom`. La sala es una arena: el jugador en
el centro y enemigos (Demons) que aparecen en los bordes y se acercan.

In [ ]:
import os
os.makedirs("scenarios", exist_ok=True)
CFG_DEFEND = "scenarios/defend_the_center.cfg"
with open(CFG_DEFEND, "w") as f:
    f.write('# Defend the Center: sala circular, enemigos aparecen en los bordes y avanzan\n# hacia el centro desde todas direcciones. Sala abierta (no pasillo).\n# Reusa el mismo layout de botones/variables que deadly_corridor para que el\n# agente entrenado (8 botones, 13 acciones) lo pueda controlar sin reentrenar.\n# El .wad (defend_the_center.wad) se inyecta desde los scenarios de ViZDoom.\n\ndoom_skill = 2\n\nscreen_resolution = RES_640X480\nscreen_format = RGB24\nwindow_visible = false\nmode = PLAYER\n\nrender_hud = true\nrender_weapon = true\nrender_crosshair = false\nrender_particles = true\nrender_decals = true\n\nliving_reward = 0.05\ndeath_penalty = 100\n\n# Mismo orden que action_to_vizdoom en policy/actions.py\n# [MOVE_FORWARD, MOVE_BACKWARD, TURN_LEFT, TURN_RIGHT, ATTACK, USE, MOVE_LEFT, MOVE_RIGHT]\navailable_buttons = {MOVE_FORWARD MOVE_BACKWARD TURN_LEFT TURN_RIGHT ATTACK USE MOVE_LEFT MOVE_RIGHT}\n\navailable_game_variables = {HEALTH AMMO2 KILLCOUNT POSITION_X}\n\nepisode_timeout = 2100\nepisode_start_time = 10\n')
print("cfg escrito:", CFG_DEFEND)


## 3. Espacio de acciones

Las 13 acciones del agente (mover, girar, disparar y combinadas). Se define primero porque
los demas modulos la referencian (`Action`).

In [ ]:
from enum import IntEnum


class Action(IntEnum):
    MOVE_FORWARD = 0
    MOVE_BACKWARD = 1
    TURN_LEFT = 2
    TURN_RIGHT = 3
    ATTACK = 4
    STRAFE_LEFT = 5          # esquiva lateral izquierda
    STRAFE_RIGHT = 6         # esquiva lateral derecha
    FORWARD_ATTACK = 7       # avanzar + disparar simultaneamente
    STRAFE_LEFT_ATTACK = 8   # esquivar izquierda + disparar
    STRAFE_RIGHT_ATTACK = 9  # esquivar derecha + disparar
    TURN_LEFT_ATTACK = 10    # girar izquierda + disparar (apuntar mientras gira)
    TURN_RIGHT_ATTACK = 11   # girar derecha + disparar
    BACKWARD_ATTACK = 12     # retroceder + disparar (kiting: mantiene distancia)


# Vector: [FORWARD, BACKWARD, TURN_L, TURN_R, ATTACK, USE, MOVE_L, MOVE_R]
def action_to_vizdoom(action: Action) -> list[int]:
    v = [0] * 8
    if action == Action.MOVE_FORWARD:
        v[0] = 1
    elif action == Action.MOVE_BACKWARD:
        v[1] = 1
    elif action == Action.TURN_LEFT:
        v[2] = 1
    elif action == Action.TURN_RIGHT:
        v[3] = 1
    elif action == Action.ATTACK:
        v[4] = 1
    elif action == Action.STRAFE_LEFT:
        v[6] = 1
    elif action == Action.STRAFE_RIGHT:
        v[7] = 1
    elif action == Action.FORWARD_ATTACK:
        v[0] = 1; v[4] = 1
    elif action == Action.STRAFE_LEFT_ATTACK:
        v[6] = 1; v[4] = 1
    elif action == Action.STRAFE_RIGHT_ATTACK:
        v[7] = 1; v[4] = 1
    elif action == Action.TURN_LEFT_ATTACK:
        v[2] = 1; v[4] = 1
    elif action == Action.TURN_RIGHT_ATTACK:
        v[3] = 1; v[4] = 1
    elif action == Action.BACKWARD_ATTACK:
        v[1] = 1; v[4] = 1
    return v


## 4. Percepcion (David)

El detector YOLO actua como extractor de percepcion. Dos piezas:

- `Detector`: corre YOLOv8 sobre el frame y devuelve las cajas. Umbral de confianza alto
  (0.40) para no alucinar en pasillos/paredes.
- `extract_state`: convierte las detecciones (+ vida, ammo) en el vector de **13 features**
  normalizadas que recibe el agente.

In [ ]:
from pathlib import Path
import torch
import numpy as np
from ultralytics import YOLO


class Detector:
    # conf alto a proposito: el detector tiene precision ~0.43, asi que un umbral
    # bajo inunda la escena de falsos positivos. 0.40 corta el grueso del ruido.
    def __init__(self, weights_path: Path, conf: float = 0.40):
        self.model = YOLO(str(weights_path))
        self.conf = conf
        # Determinar si hay GPU disponible para mover frames directamente
        self._use_gpu = torch.cuda.is_available()
        if self._use_gpu:
            self.model.to("cuda")

    def predict(self, frame):
        if frame is None:
            return None
        if self._use_gpu:
            # Convertir (H, W, 3) numpy uint8 a tensor CUDA float16 normalizado
            # Evita la transferencia CPU->GPU frame a frame dentro de YOLO
            t = torch.from_numpy(np.ascontiguousarray(frame)).cuda()
            t = t.permute(2, 0, 1).unsqueeze(0).half() / 255.0
            results = self.model.predict(t, conf=self.conf, verbose=False, half=True)
        else:
            # Ultralytics asume que un numpy HWC viene en BGR (convencion OpenCV) y
            # lo convierte a RGB internamente. ViZDoom entrega RGB24, asi que lo
            # pasamos a BGR para que esa conversion interna lo deje en RGB real.
            # Sin esto, en CPU el modelo veria los canales R y B intercambiados.
            bgr = np.ascontiguousarray(frame[:, :, ::-1])
            results = self.model.predict(bgr, conf=self.conf, verbose=False)
        return results[0]


In [ ]:
import random


# Las 10 clases del detector (referencia / dataset Roboflow).
ENEMIGOS_TODOS = {
    "imp", "cacodemon", "baron-of-hell", "cyberdemon",
    "lost-soul", "pinky", "shotgun-guy", "specter",
    "spiderdemon", "zombieman",
}

# Clases que el detector reconoce de forma fiable (AP@0.5 > 0.6 en doom-v2) y que
# aparecen en los escenarios jugados (deadly_corridor / freedoom E1M1). Las demas
# tienen AP~0 (baron-of-hell) o sin datos de validacion (cacodemon, cyberdemon,
# lost-soul, spiderdemon) y solo aportan falsos positivos, asi que las ignoramos.
ENEMIGOS = {"zombieman", "imp", "pinky", "shotgun-guy"}

# Umbral horizontal: si el enemigo esta dentro de este % del ancho, dispara.
ZONA_DISPARO = 0.20

# Contador interno para alternar exploracion (gira + avanza) sin girar en circulos.
_explorar_contador = 0
_GIROS_ANTES_AVANZAR = 6


def decidir(result, vida: int, ammo: int, frame_w: int) -> Action:
    global _explorar_contador

    centro_x = frame_w / 2
    enemigos = []

    if result is not None and len(result.boxes) > 0:
        for box in result.boxes:
            cls_name = result.names[int(box.cls[0])]
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = (x1 + x2) / 2
            area = (x2 - x1) * (y2 - y1)
            if cls_name in ENEMIGOS:
                enemigos.append((cls_name, cx, area))

    # -- Sin enemigos visibles: explorar (avanzar + giros alternados) ----------
    if not enemigos:
        _explorar_contador += 1
        if _explorar_contador % (_GIROS_ANTES_AVANZAR + 1) == 0:
            return Action.MOVE_FORWARD
        return Action.TURN_RIGHT

    _explorar_contador = 0  # reset al ver un enemigo

    # -- Con vida muy baja: retroceder buscando distancia ----------------------
    if vida < 20:
        return Action.MOVE_BACKWARD

    # -- Sin ammo: avanzar (no puede disparar) ---------------------------------
    if ammo <= 0:
        return Action.MOVE_FORWARD

    # -- Elegir el enemigo mas cercano al centro (objetivo prioritario) --------
    objetivo = min(enemigos, key=lambda e: abs(e[1] - centro_x))
    _, obj_cx, obj_area = objetivo
    distancia_x = abs(obj_cx - centro_x)
    enemigo_grande = obj_area > (frame_w * frame_w * 0.05)  # >5% del frame = cerca

    # Zona de disparo: enemigo centrado
    if distancia_x < frame_w * ZONA_DISPARO:
        # Si esta cerca Y centrado, avanza mientras dispara para no dar tregua.
        if enemigo_grande:
            return Action.ATTACK
        # Si esta lejos pero centrado, avanza para acercarse
        return Action.MOVE_FORWARD if random.random() < 0.3 else Action.ATTACK

    # Fuera de la zona de disparo: girar hacia el objetivo
    return _girar_hacia(obj_cx, centro_x)


def _girar_hacia(target_x: float, centro_x: float) -> Action:
    return Action.TURN_LEFT if target_x < centro_x else Action.TURN_RIGHT


In [ ]:
"""Puente percepcion -> estado RL.

Convierte las detecciones de YOLO (+ vida, ammo y senales de combate) en un
vector compacto de features normalizadas que el agente DQN usa como observacion.

STATE_DIM = 13:
  0  enemy_present        1 si hay >= 1 enemigo detectado
  1  nearest_offset       offset horizontal del enemigo mas centrado [-1, 1]
  2  nearest_size         area relativa del bbox mayor [0, 1]
  3  n_enemies            nº de enemigos / 5 (cap 1.0)
  4  enemy_left           1 si el enemigo mas cercano esta a la izquierda
  5  enemy_right          1 si esta a la derecha
  6  health_norm          vida / 100
  7  ammo_norm            ammo / 50 (cap 1.0)
  8  steps_no_enemy_norm  pasos sin ver enemigo / 50 (urgencia de explorar)
  9  took_damage          1 si recibio danio en el ultimo paso
  10 enemy_centered       1 si el enemigo esta en el 15% central (listo para disparar)
  11 danger_close         nearest_size si enemy_present, else 0 (amenaza inminente)
  12 enemy_distance_norm  distancia normalizada al enemigo (depth buffer); 1=lejos/desconocido
"""
import numpy as np


STATE_DIM = 13

_CENTER_THRESHOLD = 0.15  # offset normalizado dentro del cual se considera "centrado"
_MAX_DEPTH = 2000.0       # distancia maxima en unidades Doom para normalizar


def extract_state(
    result,
    health: float,
    ammo: float,
    frame_w: int,
    steps_no_enemy: int = 0,
    took_damage: float = 0.0,
    enemy_distance_norm: float = 1.0,  # 0=muy cerca, 1=lejos/desconocido
) -> np.ndarray:
    """Devuelve vector de 13 features normalizadas."""
    centro_x = frame_w / 2
    enemigos = []

    if result is not None and len(result.boxes) > 0:
        area_frame = float(frame_w * frame_w)
        for box in result.boxes:
            cls_name = result.names[int(box.cls[0])]
            if cls_name not in ENEMIGOS:
                continue
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cx = (x1 + x2) / 2
            area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
            enemigos.append((cx - centro_x, area / area_frame, cx))

    if enemigos:
        offset, _, cx = min(enemigos, key=lambda e: abs(e[0]))
        nearest_offset  = float(np.clip(offset / centro_x, -1.0, 1.0))
        nearest_size    = float(np.clip(max(e[1] for e in enemigos), 0.0, 1.0))
        n_enemies       = float(np.clip(len(enemigos) / 5.0, 0.0, 1.0))
        enemy_left      = 1.0 if cx < centro_x else 0.0
        enemy_right     = 1.0 if cx >= centro_x else 0.0
        enemy_present   = 1.0
        enemy_centered  = 1.0 if abs(nearest_offset) < _CENTER_THRESHOLD else 0.0
        danger_close    = nearest_size  # ya normalizado [0,1]
    else:
        nearest_offset = nearest_size = n_enemies = 0.0
        enemy_left = enemy_right = enemy_present = 0.0
        enemy_centered = danger_close = 0.0

    health_norm         = float(np.clip(health / 100.0, 0.0, 1.0))
    ammo_norm           = float(np.clip(ammo / 50.0, 0.0, 1.0))
    steps_no_enemy_norm = float(np.clip(steps_no_enemy / 50.0, 0.0, 1.0))
    took_damage_f       = float(np.clip(took_damage, 0.0, 1.0))

    return np.array(
        [enemy_present, nearest_offset, nearest_size, n_enemies,
         enemy_left, enemy_right, health_norm, ammo_norm,
         steps_no_enemy_norm, took_damage_f,
         enemy_centered, danger_close,
         float(np.clip(enemy_distance_norm, 0.0, 1.0))],
        dtype=np.float32,
    )


In [ ]:
import cv2


def draw_detections(frame, result):
    img = frame.copy()
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        label = f"{result.names[cls]} {conf:.2f}"
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, label, (x1, y1 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    return img


## 5. Detector YOLO (percepcion)

El detector se entreno **en LOCAL** (no aqui), para ahorrar tiempo de GPU en Kaggle. Esta
celda **carga los pesos ya entrenados** (`doom-v4`) clonando el repositorio publico del
proyecto. El proceso completo de como se construyo se documenta justo despues, pero **esas
celdas son de referencia y no se ejecutan**.

**Metodo (resumen):** ViZDoom expone un `labels_buffer` con la posicion y el nombre de cada
objeto en pantalla. Capturamos frames de varios escenarios y sacamos las cajas ground-truth
directo del motor (sin etiquetar a mano), mas frames vacios como fondo. Entrenamos YOLOv8s
sobre ese dataset in-domain. Resultado en local: mAP@0.5 ~0.91, deteccion de pinky en la
sala ~0.89.

In [ ]:
from pathlib import Path
# Detector v4 entrenado en LOCAL. El repo es publico: clonamos (shallow) solo para
# traer los pesos ya entrenados. El codigo de como se entreno esta documentado abajo.
!git clone -q --depth 1 https://github.com/josh0827/IA_Doom_agent.git _repo
DETECTOR_WEIGHTS = Path("_repo/runs/doom-v4/weights/best.pt")
assert DETECTOR_WEIGHTS.exists(), "No se encontro best.pt tras clonar el repo."
print("detector v4 (entrenado en LOCAL):", DETECTOR_WEIGHTS,
      DETECTOR_WEIGHTS.stat().st_size // 1024, "KB")


### Como se construyo el detector (ejecutado en LOCAL, no aqui)

**1) Captura + auto-etiquetado desde ViZDoom** (cajas ground-truth, sin etiquetar a mano):

```python
import random, glob
import cv2
import numpy as np
import vizdoom as vzd

CLASSES = ["baron-of-hell","cacodemon","cyberdemon","imp","lost-soul",
           "pinky","shotgun-guy","specter","spiderdemon","zombieman"]
CIDX = {c: i for i, c in enumerate(CLASSES)}
NAME_MAP = {"Zombieman":"zombieman","ShotgunGuy":"shotgun-guy","ChaingunGuy":"shotgun-guy",
            "DoomImp":"imp","Demon":"pinky","Spectre":"specter","Cacodemon":"cacodemon",
            "BaronOfHell":"baron-of-hell","LostSoul":"lost-soul","Cyberdemon":"cyberdemon",
            "SpiderMastermind":"spiderdemon"}
SP = __import__("pathlib").Path(vzd.scenarios_path)   # cfg/wad que trae el paquete vizdoom
CAP_SCEN = ["deadly_corridor", "defend_the_center", "basic"]
MIN_BOX, EMPTY_RATE = 10, 0.20
DATA_DIR = __import__("pathlib").Path("dataset/doom-vizdoom")

def make_cap_game(name):
    g = vzd.DoomGame(); g.load_config(str(SP / f"{name}.cfg"))
    g.set_doom_scenario_path(str(SP / f"{name}.wad"))
    g.set_doom_skill(3); g.set_window_visible(False)
    g.set_screen_format(vzd.ScreenFormat.RGB24)
    g.set_screen_resolution(vzd.ScreenResolution.RES_640X480)
    g.set_labels_buffer_enabled(True); g.set_render_hud(False); g.init(); return g

def boxes_from_labels(st, w, h):
    out = []
    for l in st.labels:
        c = NAME_MAP.get(l.object_name)
        if c is None or l.width < MIN_BOX or l.height < MIN_BOX:
            continue
        out.append((CIDX[c], (l.x+l.width/2)/w, (l.y+l.height/2)/h, l.width/w, l.height/h))
    return out

def capturar(frames_por_scen=420, val_split=0.15, seed=0):
    random.seed(seed)
    for sp in ("train","valid"):
        for sub in ("images","labels"):
            (DATA_DIR/sp/sub).mkdir(parents=True, exist_ok=True)
    saved, counts = 0, {c:0 for c in CLASSES}
    for name in CAP_SCEN:
        try:
            g = make_cap_game(name)
        except Exception as e:
            print("salto", name, e); continue
        nb = g.get_available_buttons_size(); g.new_episode(); got = 0; tries = 0
        while got < frames_por_scen and tries < frames_por_scen*40:
            tries += 1
            if g.is_episode_finished(): g.new_episode()
            a = [0]*nb
            for b in random.sample(range(nb), k=min(2,nb)): a[b] = random.randint(0,1)
            g.make_action(a, random.choice([2,3,4]))
            st = g.get_state()
            if st is None: continue
            fr = st.screen_buffer; h,w = fr.shape[:2]
            bx = boxes_from_labels(st, w, h)
            if not bx and random.random() > EMPTY_RATE: continue
            split = "valid" if random.random() < val_split else "train"
            stem = f"{name}_{saved:05d}"
            cv2.imwrite(str(DATA_DIR/split/"images"/f"{stem}.jpg"), cv2.cvtColor(fr, cv2.COLOR_RGB2BGR))
            with open(DATA_DIR/split/"labels"/f"{stem}.txt","w") as f:
                for ci,cx,cy,bw,bh in bx:
                    f.write(f"{ci} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n"); counts[CLASSES[ci]] += 1
            saved += 1; got += 1
        g.close(); print(f"{name}: {got} frames")
    with open(DATA_DIR/"data.yaml","w") as f:
        f.write("train: ../train/images\nval: ../valid/images\n")
        f.write(f"nc: {len(CLASSES)}\nnames: {CLASSES}\n")
    print("total", saved, "| por clase:", {k:v for k,v in counts.items() if v})

capturar()
```

**2) Entrenamiento del detector YOLOv8s:**

```python
from ultralytics import YOLO

det_model = YOLO("yolov8s.pt")
det_model.train(data=str(DATA_DIR/"data.yaml"), epochs=60, imgsz=640, batch=16,
                workers=2, cos_lr=True, patience=20, device=0,
                project="runs", name="doom-v4", exist_ok=True, verbose=True)
DETECTOR_WEIGHTS = Path("runs/doom-v4/weights/best.pt")
```

## 6. Entorno (Joshua)

- `DoomEnv`: wrapper crudo de ViZDoom (frame, vida, ammo, kills, posicion, fin de nivel).
- `RLEnv`: une ViZDoom + detector + features y define la **recompensa**. Es *scenario-aware*:
  en la sala (`defend_the_center`) premia **sobrevivir, encarar y barrer girando, y matar**,
  y penaliza disparar a la nada (cuidar municion).
- `FrameStack`: apila 3 vectores para dar memoria de corto plazo (39 dims).

In [ ]:
import os
from pathlib import Path
import vizdoom as vzd


class DoomEnv:
    def __init__(self, scenario_cfg: Path, window_visible: bool = False, skill: int | None = None):
        self.game = vzd.DoomGame()
        self.game.load_config(str(scenario_cfg))
        # Inyecta el .wad buscando en este orden:
        # 1. <scenarios_dir>/<stem>.wad  (escenarios de ViZDoom, e.g. deadly_corridor)
        # 2. <mismo directorio que el cfg>/<stem>.wad  (WADs propios, e.g. freedoom1)
        # 3. <mismo directorio que el cfg>/freedoom1.wad  (alias para cfg "freedoom")
        scenarios_dir = Path(vzd.scenarios_path)
        cfg_dir = scenario_cfg.parent
        candidates = [
            scenarios_dir / f"{scenario_cfg.stem}.wad",
            cfg_dir / f"{scenario_cfg.stem}.wad",
            cfg_dir / "freedoom1.wad",
        ]
        for wad in candidates:
            if wad.exists():
                self.game.set_doom_scenario_path(str(wad))
                break
        if skill is not None:
            self.game.set_doom_skill(skill)
        self.game.set_window_visible(window_visible)
        self.game.set_screen_format(vzd.ScreenFormat.RGB24)
        self.game.set_depth_buffer_enabled(True)
        # Renderizado por GPU (OpenGL): acelera frames con depth buffer activo.
        # En servidores sin pantalla (Kaggle/Colab) OpenGL falla -> con DOOM_HEADLESS
        # se omite y ViZDoom usa el software renderer headless.
        if not os.environ.get("DOOM_HEADLESS"):
            self.game.add_game_args("+vid_renderer opengl +gl_multisample 0")
        self.game.init()
        # POSITION_X disponible solo si el cfg lo declara (deadly_corridor): permite
        # medir avance real para la recompensa de progreso potencial.
        self._has_posx = vzd.GameVariable.POSITION_X in self.game.get_available_game_variables()
        self._last_info = {"vida": 100, "ammo": 0, "kills": 0, "pos_x": 0.0}
        self._last_depth = None

    def reset(self):
        self._last_info = {"vida": 100, "ammo": 0, "kills": 0}
        self.game.new_episode()
        return self._frame()

    def step(self, action_vector: list[int], tics: int = 1):
        # tics > 1 aplica la misma accion varios frames (frame-skip).
        reward = self.game.make_action(action_vector, tics)
        done = self.game.is_episode_finished()
        frame = None if done else self._frame()
        state = self.game.get_state()
        if state:
            self._last_info = {
                "vida":  self.game.get_game_variable(vzd.GameVariable.HEALTH),
                "ammo":  self.game.get_game_variable(vzd.GameVariable.AMMO2),
                "kills": self.game.get_game_variable(vzd.GameVariable.KILLCOUNT),
                "pos_x": (self.game.get_game_variable(vzd.GameVariable.POSITION_X)
                          if self._has_posx else 0.0),
            }
        # Al morir (state=None) devuelve el ultimo valor conocido para no perder kills.
        info = dict(self._last_info)
        dead = self.game.is_player_dead()
        info["dead"] = dead  # fiable aunque state sea None
        # Completo el nivel = termino vivo y ANTES del timeout (alcanzo el chaleco).
        # Si agota el tiempo, get_episode_time() == get_episode_timeout().
        info["completed"] = bool(
            done and not dead
            and self.game.get_episode_time() < self.game.get_episode_timeout()
        )
        return frame, reward, done, info

    def _frame(self):
        state = self.game.get_state()
        if state is None:
            self._last_depth = None
            return None
        self._last_depth = state.depth_buffer  # (H, W) float32, distancia en unidades Doom
        return state.screen_buffer  # ViZDoom 1.3+ devuelve (H, W, 3) directamente

    @property
    def depth_buffer(self):
        return self._last_depth

    def close(self):
        self.game.close()


In [ ]:
"""Entorno RL: envuelve ViZDoom + detector YOLO.

Observacion: vector de 13 features (ver perception/features.py).

Recompensa (shaping):
  + KILL_REWARD     por cada baja confirmada (objetivo principal)
  + SHOOT_REWARD      senal densa por disparar cuando hay enemigo visible
  - WASTE_SHOT_PENALTY penaliza disparar sin enemigo visible (dispara a la nada)
  + STRAFE_REWARD   bonus por esquivar (strafe) cuando hay enemigo a la vista
  - HEALTH_PENALTY  penaliza recibir danio (delta de vida negativo)
  - DEATH_PENALTY   castigo explicito por morir
  - LIVING_COST     costo por paso: desincentiva merodear/farmear tiempo
  + PROGRESS_PER_UNIT  progreso POTENCIAL: premia solo terreno NUEVO ganado hacia
                    el chaleco (delta de la X maxima). NO es velocidad, asi que
                    merodear no farmea reward. Acotado por el largo del corredor.
  + GOAL_REWARD     premio terminal por COMPLETAR el nivel (alcanzar el chaleco
                    vivo, antes del timeout). Domina sobre cualquier farmeo.

Espacio de accion: 13 acciones (incluye strafe y acciones combinadas).
  0 MOVE_FORWARD        5 STRAFE_LEFT          10 TURN_LEFT_ATTACK
  1 MOVE_BACKWARD       6 STRAFE_RIGHT         11 TURN_RIGHT_ATTACK
  2 TURN_LEFT           7 FORWARD_ATTACK       12 BACKWARD_ATTACK
  3 TURN_RIGHT          8 STRAFE_LEFT_ATTACK
  4 ATTACK              9 STRAFE_RIGHT_ATTACK
"""
from pathlib import Path

import numpy as np


N_ACTIONS = 13

KILL_REWARD       = 150.0
SHOOT_REWARD      = 2.0    # disparar CON enemigo visible
WASTE_SHOT_PENALTY = 5.0   # disparar SIN enemigo visible (antes 1.5: rociaba paredes gratis)
STRAFE_REWARD     = 0.8    # esquivar con enemigo visible
HEALTH_PENALTY    = 0.8
DEATH_PENALTY     = 100.0  # morir duele (antes 50: rushear-y-morir salia rentable)
LIVING_COST       = 0.05   # costo por paso: desincentiva merodear/farmear tiempo
GOAL_REWARD       = 300.0  # completar el nivel: claramente mas que cualquier farmeo

# Progreso POTENCIAL: se premia solo el terreno NUEVO ganado hacia el chaleco
# (delta de la X maxima alcanzada), NO la velocidad de movimiento. Asi merodear o
# ir y venir NO acumula reward (no farmeable). Corredor mide ~1280 unidades en X,
# asi que un recorrido completo aporta ~1280 * PROGRESS_PER_UNIT.
PROGRESS_PER_UNIT = 0.2    # 1280 ud -> ~256 de progreso total (comparable a ~1.7 kills)

# ── Reward de SALA ABIERTA (defend_the_center, torreta) ────────────────────────
# Objetivo distinto: no hay chaleco/avance; hay que sobrevivir girando y matando
# enemigos que llegan de todos lados, cuidando municion.
SURVIVAL_BONUS = 0.1   # por paso vivo (en sala SI premiamos durar; sustituye LIVING_COST)
AIM_REWARD     = 0.2   # enemigo centrado (encararlo) -> fomenta apuntar girando
SCAN_REWARD    = 0.05  # girar cuando NO hay enemigo a la vista -> barrido izq-der
GOAL_ROOM      = 50.0  # sobrevivir hasta el timeout (aguantar toda la ronda)

# Confianza minima para que una deteccion cuente como "enemigo de combate" en la
# recompensa. Mas estricta que el umbral del detector (0.40): evita que un falso
# positivo marginal haga que disparar a la nada cobre SHOOT_REWARD en lugar de
# pagar WASTE_SHOT_PENALTY. Es el dial que rompe el "disparar a fantasmas".
COMBAT_CONF = 0.5

_STRAFE_ACTIONS = {Action.STRAFE_LEFT, Action.STRAFE_RIGHT,
                   Action.STRAFE_LEFT_ATTACK, Action.STRAFE_RIGHT_ATTACK,
                   Action.BACKWARD_ATTACK}  # kiting tambien es evasion
_SHOOT_ACTIONS  = {Action.ATTACK, Action.FORWARD_ATTACK,
                   Action.STRAFE_LEFT_ATTACK, Action.STRAFE_RIGHT_ATTACK,
                   Action.TURN_LEFT_ATTACK, Action.TURN_RIGHT_ATTACK,
                   Action.BACKWARD_ATTACK}
_TURN_ACTIONS   = {Action.TURN_LEFT, Action.TURN_RIGHT,
                   Action.TURN_LEFT_ATTACK, Action.TURN_RIGHT_ATTACK}

_MAX_DEPTH = 1000.0  # combate tipico ocurre a 200-800 unidades, mejor resolucion


class RLEnv:
    def __init__(
        self,
        weights: Path,
        scenario: Path,
        frame_skip: int = 2,
        conf: float = 0.40,  # ver nota en Detector: umbral alto contra falsos positivos
        window_visible: bool = False,
        skill: int | None = None,
    ):
        self.env = DoomEnv(scenario, window_visible=window_visible, skill=skill)
        self.detector = Detector(weights, conf=conf)
        # Sala abierta (defend_the_center) usa reward de torreta; pasillo usa progreso+meta.
        self._room = "defend" in Path(scenario).stem.lower()
        self.frame_skip = frame_skip
        self.state_dim = STATE_DIM
        self.n_actions = N_ACTIONS
        self._last_overlay_data = None
        self._prev_health = 100.0
        self._prev_kills  = 0.0
        self._steps_no_enemy = 0
        self._max_x = 0.0    # X maxima alcanzada (progreso potencial, no farmeable)
        self._frame_w = 640  # se actualiza en reset

    def reset(self) -> np.ndarray:
        frame = self.env.reset()
        self._prev_health    = self.env._last_info["vida"]
        self._prev_kills     = self.env._last_info["kills"]
        self._steps_no_enemy = 0
        self._max_x          = self.env._last_info.get("pos_x", 0.0)
        ammo = self.env._last_info["ammo"]

        if frame is not None:
            self._frame_w = frame.shape[1]

        result = self.detector.predict(frame)
        self._last_overlay_data = (frame, result)
        return extract_state(result, self._prev_health, ammo,
                             self._frame_w, self._steps_no_enemy, 0.0,
                             self._enemy_distance(result))

    def step(self, action_idx: int):
        action = Action(action_idx)
        frame, reward_env, done, info = self.env.step(
            action_to_vizdoom(action), tics=self.frame_skip
        )
        # No usamos el reward nativo (premia velocidad => farmeable). Construimos
        # el shaping desde cero con progreso potencial + eventos.
        reward = 0.0

        if done or frame is None:
            if info.get("dead"):
                reward -= DEATH_PENALTY
            elif info.get("completed"):
                reward += GOAL_ROOM if self._room else GOAL_REWARD
            self._last_overlay_data = None
            state = np.zeros(self.state_dim, dtype=np.float32)
            return state, reward, True, info

        vida, ammo, kills = info["vida"], info["ammo"], info["kills"]
        delta_kills  = max(0.0, kills - self._prev_kills)
        delta_health = vida - self._prev_health           # negativo si recibio danio
        took_damage  = 1.0 if delta_health < 0 else 0.0

        # ── Percepcion: enemigo a la vista (conf alta) y centrado? ─────────────
        result = self.detector.predict(frame)
        self._last_overlay_data = (frame, result)
        centro_x = self._frame_w / 2
        enemy_present = enemy_centered = False
        if result is not None and len(result.boxes) > 0:
            for box in result.boxes:
                if (result.names[int(box.cls[0])] in ENEMIGOS
                        and float(box.conf[0]) >= COMBAT_CONF):
                    enemy_present = True
                    x1, _, x2, _ = box.xyxy[0].tolist()
                    if abs((x1 + x2) / 2 - centro_x) < 0.15 * self._frame_w:
                        enemy_centered = True

        # ── Eventos comunes (pasillo y sala) ───────────────────────────────────
        reward += KILL_REWARD * delta_kills
        reward += HEALTH_PENALTY * delta_health            # penaliza danio recibido
        if enemy_present and action in _SHOOT_ACTIONS:
            reward += SHOOT_REWARD                          # disparar a blanco real
        if not enemy_present and action in _SHOOT_ACTIONS:
            reward -= WASTE_SHOT_PENALTY                    # disparar a la nada
        if enemy_present and action in _STRAFE_ACTIONS:
            reward += STRAFE_REWARD                         # esquivar

        # ── Shaping especifico del modo ────────────────────────────────────────
        if self._room:
            reward += SURVIVAL_BONUS                        # sala: durar tiene valor
            if enemy_present and enemy_centered:
                reward += AIM_REWARD                        # encarar al enemigo
            if not enemy_present and action in _TURN_ACTIONS:
                reward += SCAN_REWARD                       # barrido buscando amenazas
        else:
            reward -= LIVING_COST                           # pasillo: no merodear
            pos_x = info.get("pos_x", 0.0)                  # progreso potencial (X nueva)
            if pos_x > self._max_x:
                reward += PROGRESS_PER_UNIT * (pos_x - self._max_x)
                self._max_x = pos_x

        self._steps_no_enemy = 0 if enemy_present else self._steps_no_enemy + 1
        self._prev_kills  = kills
        self._prev_health = vida

        next_state = extract_state(
            result, vida, ammo, self._frame_w,
            self._steps_no_enemy, took_damage,
            self._enemy_distance(result),
        )
        return next_state, reward, False, info

    def _enemy_distance(self, result) -> float:
        """Distancia normalizada al enemigo mas cercano usando el depth buffer."""
        depth = self.env.depth_buffer
        if depth is None or result is None or len(result.boxes) == 0:
            return 1.0  # desconocido = lejos
        centro_x = self._frame_w / 2
        best_cx, best_dist = None, float("inf")
        for box in result.boxes:
            if result.names[int(box.cls[0])] not in ENEMIGOS:
                continue
            x1, _, x2, _ = box.xyxy[0].tolist()
            cx = (x1 + x2) / 2
            if abs(cx - centro_x) < best_dist:
                best_dist = abs(cx - centro_x)
                best_cx = int(np.clip(cx, 0, depth.shape[1] - 1))
        if best_cx is None:
            return 1.0
        cy = depth.shape[0] // 2
        dist = float(depth[cy, best_cx])
        return float(np.clip(dist / _MAX_DEPTH, 0.0, 1.0))

    @property
    def last_overlay_data(self):
        return self._last_overlay_data

    def close(self):
        self.env.close()


In [ ]:
"""FrameStack: apila N vectores de estado consecutivos.

Sin esto el agente solo ve el momento actual y no puede saber si un enemigo
se acerca, si su vida esta cayendo rapido, o si acaba de girar. Apilar 3 frames
le da memoria de corto plazo sin la complejidad de una red recurrente (LSTM).

El input al DQN pasa de STATE_DIM a STATE_DIM * N_FRAMES (ej: 13*3 = 39).
Los frames mas nuevos van al final del vector.
"""
from collections import deque

import numpy as np


class FrameStack:
    def __init__(self, env, n_frames: int = 3):
        self.env = env
        self.n_frames = n_frames
        self._frames = deque(maxlen=n_frames)
        self.state_dim = env.state_dim * n_frames
        self.n_actions = env.n_actions

    def reset(self) -> np.ndarray:
        state = self.env.reset()
        # Rellena con el primer estado para que no haya ceros al inicio
        for _ in range(self.n_frames):
            self._frames.append(state)
        return self._obs()

    def step(self, action: int):
        state, reward, done, info = self.env.step(action)
        self._frames.append(state)
        return self._obs(), reward, done, info

    def _obs(self) -> np.ndarray:
        return np.concatenate(list(self._frames), dtype=np.float32)

    @property
    def last_overlay_data(self):
        return self.env.last_overlay_data

    def close(self):
        self.env.close()


## 7. Politica: Double DQN (Dueling + PER + n-step)

- `QNetwork`: MLP Dueling (separa V(s) y A(s,a)).
- `PrioritizedReplayBuffer`: muestrea las transiciones con mayor error TD.
- `DQNAgent`: Double DQN. `act(forbidden=...)` permite enmascarar acciones (en la sala
  prohibimos avanzar: el agente es una torreta que gira y dispara).

In [ ]:
"""Componentes del DQN: red Q (Dueling MLP) y replay buffer.

Dueling architecture (Wang et al. 2016):
  Q(s,a) = V(s) + A(s,a) - mean_a[A(s,a)]

Separar valor del estado (V) y ventaja por accion (A) acelera el aprendizaje
porque el agente puede actualizar V aunque no haya ejecutado todas las acciones.
"""
import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn


class QNetwork(nn.Module):
    """Dueling MLP: tronco compartido -> cabezas Value y Advantage."""

    def __init__(self, state_dim: int, n_actions: int, hidden: int = 512):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
        )
        self.value = nn.Sequential(
            nn.Linear(hidden // 2, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
        )
        self.advantage = nn.Sequential(
            nn.Linear(hidden // 2, 256),
            nn.ReLU(),
            nn.Linear(256, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.shared(x)
        v = self.value(features)
        a = self.advantage(features)
        # Q = V + A - mean(A)  =>  las Q siguen siendo comparables entre acciones
        return v + a - a.mean(dim=1, keepdim=True)


class ReplayBuffer:
    """Buffer uniforme — se conserva como fallback y para tests."""

    def __init__(self, capacity: int = 100_000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done, gamma_n):
        self.buffer.append((state, action, reward, next_state, done, gamma_n))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones, gammas = zip(*batch)
        return (
            np.array(states, dtype=np.float32),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones, dtype=np.float32),
            np.array(gammas, dtype=np.float32),
        )

    def __len__(self) -> int:
        return len(self.buffer)


# ── Prioritized Experience Replay ─────────────────────────────────────────────

class _SumTree:
    """Arbol binario de sumas para muestreo O(log N) proporcional a prioridades."""

    def __init__(self, capacity: int):
        self.capacity = capacity
        self.tree = np.zeros(2 * capacity - 1, dtype=np.float64)
        self.data: list = [None] * capacity
        self.n_entries = 0
        self._write = 0

    def _propagate(self, idx: int, delta: float):
        parent = (idx - 1) // 2
        self.tree[parent] += delta
        if parent:
            self._propagate(parent, delta)

    def _retrieve(self, idx: int, s: float) -> int:
        left = 2 * idx + 1
        if left >= len(self.tree):
            return idx
        return self._retrieve(left, s) if s <= self.tree[left] else self._retrieve(left + 1, s - self.tree[left])

    @property
    def total(self) -> float:
        return float(self.tree[0])

    def add(self, priority: float, data):
        idx = self._write + self.capacity - 1
        self.data[self._write] = data
        self.update(idx, priority)
        self._write = (self._write + 1) % self.capacity
        self.n_entries = min(self.n_entries + 1, self.capacity)

    def update(self, idx: int, priority: float):
        self._propagate(idx, priority - self.tree[idx])
        self.tree[idx] = priority

    def get(self, s: float):
        idx = self._retrieve(0, s)
        data_idx = idx - self.capacity + 1
        return idx, float(self.tree[idx]), self.data[data_idx]


class PrioritizedReplayBuffer:
    """PER (Schaul et al. 2016): muestrea transiciones con mayor TD-error mas frecuentemente.

    alpha  controla cuanto afecta la prioridad (0 = uniforme, 1 = puro PER).
    beta   corrige el sesgo de muestreo (importance sampling); sube de beta_start a 1.0.
    """

    def __init__(
        self,
        capacity: int = 100_000,
        alpha: float = 0.6,
        beta_start: float = 0.4,
        beta_frames: int = 80_000,
    ):
        self.tree = _SumTree(capacity)
        self.alpha = alpha
        self.beta_start = beta_start
        self.beta_frames = beta_frames
        self._frame = 0
        self._max_priority = 1.0

    def beta(self) -> float:
        return min(1.0, self.beta_start + self._frame * (1.0 - self.beta_start) / self.beta_frames)

    def push(self, state, action, reward, next_state, done, gamma_n):
        self.tree.add(self._max_priority ** self.alpha,
                      (state, action, reward, next_state, done, gamma_n))

    def sample(self, batch_size: int):
        self._frame += 1
        segment = self.tree.total / batch_size
        batch, idxs, priorities = [], [], []
        for i in range(batch_size):
            s = np.random.uniform(segment * i, segment * (i + 1))
            idx, p, data = self.tree.get(s)
            if data is None:
                data = self.tree.data[0]
                idx = self.tree.capacity - 1
                p = max(self.tree.tree[idx], 1e-8)
            batch.append(data)
            idxs.append(idx)
            priorities.append(p)
        probs = np.array(priorities) / (self.tree.total + 1e-8)
        weights = (self.tree.n_entries * probs) ** (-self.beta())
        weights /= weights.max()
        states, actions, rewards, next_states, dones, gammas = zip(*batch)
        return (
            np.array(states,      dtype=np.float32),
            np.array(actions,     dtype=np.int64),
            np.array(rewards,     dtype=np.float32),
            np.array(next_states, dtype=np.float32),
            np.array(dones,       dtype=np.float32),
            np.array(gammas,      dtype=np.float32),
            idxs,
            np.array(weights,     dtype=np.float32),
        )

    def update_priorities(self, idxs: list, td_errors: np.ndarray):
        for idx, err in zip(idxs, td_errors):
            p = float(abs(err)) + 1e-6
            self._max_priority = max(self._max_priority, p)
            self.tree.update(idx, (p ** self.alpha))

    def __len__(self) -> int:
        return self.tree.n_entries


In [ ]:
"""Agente Double DQN con Dueling architecture.

Mejoras respecto al DQN vanilla:
  - Double DQN: policy_net selecciona la accion, target_net evalua su Q-value.
    Esto corrige la sobreestimacion sistematica del DQN original.
  - Dueling QNetwork: separa V(s) y A(s,a), aprende mas rapido porque actualiza
    el valor del estado aunque no haya ejecutado todas las acciones.
  - Buffer 100k, lr 5e-4, eps_decay 50k pasos, n-step returns (gamma^N en el target).
"""
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn



class DQNAgent:
    def __init__(
        self,
        state_dim: int,
        n_actions: int,
        lr: float = 5e-4,
        gamma: float = 0.99,
        batch_size: int = 512,
        buffer_capacity: int = 100_000,
        target_update: int = 2000,
        eps_start: float = 1.0,
        eps_end: float = 0.05,
        eps_decay_steps: int = 50_000,
        device: str | None = None,
    ):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update = target_update
        self.eps_start = eps_start
        self.eps_end = eps_end
        self.eps_decay_steps = eps_decay_steps

        self.policy_net = QNetwork(state_dim, n_actions).to(self.device)
        self.target_net = QNetwork(state_dim, n_actions).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = torch.optim.Adam(self.policy_net.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=eps_decay_steps * 2, eta_min=lr * 0.1
        )
        self.buffer = PrioritizedReplayBuffer(buffer_capacity)
        self.steps = 0

    def epsilon(self) -> float:
        frac = min(1.0, self.steps / self.eps_decay_steps)
        return self.eps_start + frac * (self.eps_end - self.eps_start)

    def act(self, state: np.ndarray, greedy: bool = False, forbidden=None) -> int:
        """forbidden: set de indices de accion prohibidos (p. ej. avanzar en sala)."""
        allowed = ([a for a in range(self.n_actions) if a not in forbidden]
                   if forbidden else None)
        if not greedy and np.random.rand() < self.epsilon():
            return int(np.random.choice(allowed)) if allowed else np.random.randint(self.n_actions)
        with torch.no_grad():
            s = torch.as_tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            q = self.policy_net(s).squeeze(0)
            if forbidden:
                for i in forbidden:
                    q[i] = -1e9
            return int(q.argmax().item())

    def learn(self) -> float | None:
        """Double DQN con PER: loss ponderada por importance sampling."""
        if len(self.buffer) < self.batch_size:
            return None

        states, actions, rewards, next_states, dones, gammas, idxs, weights = \
            self.buffer.sample(self.batch_size)

        states      = torch.as_tensor(states,      device=self.device)
        actions     = torch.as_tensor(actions,     device=self.device).unsqueeze(1)
        rewards     = torch.as_tensor(rewards,     device=self.device).unsqueeze(1)
        next_states = torch.as_tensor(next_states, device=self.device)
        dones       = torch.as_tensor(dones,       device=self.device).unsqueeze(1)
        gammas      = torch.as_tensor(gammas,      device=self.device).unsqueeze(1)
        weights     = torch.as_tensor(weights,     device=self.device).unsqueeze(1)

        q = self.policy_net(states).gather(1, actions)

        with torch.no_grad():
            best_actions = self.policy_net(next_states).argmax(dim=1, keepdim=True)
            q_next = self.target_net(next_states).gather(1, best_actions)
            # gammas ya es gamma^N (N = pasos del retorno n-step de cada transicion),
            # el descuento correcto para arrancar el valor bootstrapeado.
            target = rewards + gammas * q_next * (1.0 - dones)

        td_errors = (q - target).abs().detach().cpu().numpy().squeeze()
        # Loss ponderada por IS weights para corregir el sesgo del PER
        loss = (weights * nn.functional.smooth_l1_loss(q, target, reduction="none")).mean()

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optimizer.step()

        self.buffer.update_priorities(idxs, td_errors)

        self.steps += 1
        self.scheduler.step()
        if self.steps % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return float(loss.item())

    def save(self, path: Path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        torch.save(self.policy_net.state_dict(), str(path))

    def load(self, path: Path):
        state = torch.load(str(path), map_location=self.device)
        self.policy_net.load_state_dict(state)
        self.target_net.load_state_dict(state)


## 8. Entrenamiento del agente

Bucle de entrenamiento con n-step returns y evaluacion greedy periodica. Mascara
**sin avanzar** (torreta). Ajusta `TIMESTEPS` segun el tiempo disponible (el limite de
Kaggle es 12 h; con el detector corriendo por paso, ~1M pasos caben). Guarda el mejor
modelo por evaluacion en `dqn_room.pt`.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from collections import deque

TIMESTEPS  = 1_000_000     # presupuesto de pasos (bajalo si tu sesion es corta)
MAX_STEPS  = 1500
FRAME_SKIP = 2
N_STEP     = 3
forbidden  = {int(Action.MOVE_FORWARD), int(Action.FORWARD_ATTACK)}  # torreta: no avanzar

class NStepBuffer:
    def __init__(self, n, gamma): self.n, self.gamma, self._b = n, gamma, deque()
    def push(self, s,a,r,s2,d):
        self._b.append((s,a,r,s2,d))
        return self._emit() if len(self._b) >= self.n else None
    def flush(self):
        out = []
        while self._b: out.append(self._emit())
        return out
    def _emit(self):
        s0,a0 = self._b[0][0], self._b[0][1]; G,gn = 0.0,1.0; sn,dn = self._b[-1][3], False
        for _,_,r,s2,d in self._b:
            G += gn*r; gn *= self.gamma
            if d: sn,dn = s2,True; break
        self._b.popleft(); return s0,a0,G,sn,float(dn),gn

def make_env():
    return FrameStack(RLEnv(DETECTOR_WEIGHTS, Path(CFG_DEFEND), frame_skip=FRAME_SKIP), n_frames=3)

def evaluate(agent, n_ep=3):
    R,K = [],[]
    for _ in range(n_ep):
        e = make_env(); s = e.reset(); tot,info = 0.0,{}
        for _ in range(MAX_STEPS):
            s,r,d,info = e.step(agent.act(s, greedy=True, forbidden=forbidden)); tot += r
            if d: break
        e.close(); R.append(tot); K.append(int(info.get("kills",0)))
    return float(np.mean(R)), float(np.mean(K))

env = make_env()
agent = DQNAgent(env.state_dim, env.n_actions, device="cuda" if torch.cuda.is_available() else "cpu")
nstep = NStepBuffer(N_STEP, agent.gamma)
rewards_hist, recientes, kills_rec = [], deque(maxlen=20), deque(maxlen=20)
mejor, total_steps, ep = float("-inf"), 0, 0
CKPT = Path("runs/rl/dqn_room.pt"); CKPT.parent.mkdir(parents=True, exist_ok=True)

while total_steps < TIMESTEPS:
    ep += 1; s = env.reset(); tot, losses, ep_kills = 0.0, [], 0
    for _ in range(MAX_STEPS):
        a = agent.act(s, forbidden=forbidden)
        s2,r,d,info = env.step(a); total_steps += 1
        tr = nstep.push(s,a,r,s2,float(d))
        if tr: agent.buffer.push(*tr)
        loss = agent.learn()
        if loss is not None: losses.append(loss)
        s = s2; tot += r; ep_kills = int(info.get("kills",0))
        if d: break
    for t in nstep.flush(): agent.buffer.push(*t)
    rewards_hist.append(tot); recientes.append(tot); kills_rec.append(ep_kills)
    if ep % 10 == 0:
        ls = f"{np.mean(losses):.3f}" if losses else "N/A"
        print(f"pasos {total_steps}/{TIMESTEPS} | ep {ep} | reward {tot:7.1f} | "
              f"media20 {np.mean(recientes):7.1f} | kills {ep_kills} | "
              f"kills_avg {np.mean(kills_rec):.1f} | eps {agent.epsilon():.3f} | loss {ls}")
    if ep % 50 == 0:
        er, ek = evaluate(agent)
        print(f"  [EVAL] reward {er:.1f} | kills {ek:.1f}")
        if er > mejor:
            mejor = er; agent.save(CKPT); print(f"  [CKPT] mejor guardado ({er:.1f})")
env.close()
plt.figure(figsize=(9,5)); plt.plot(rewards_hist, alpha=.35)
if len(rewards_hist) >= 20:
    mv = [np.mean(rewards_hist[max(0,i-19):i+1]) for i in range(len(rewards_hist))]
    plt.plot(mv, "crimson", lw=2)
plt.xlabel("Episodio"); plt.ylabel("Recompensa"); plt.title("Curva de aprendizaje (sala)")
plt.grid(alpha=.3); plt.tight_layout(); plt.savefig("runs/rl/learning_curve_room.png", dpi=120)
plt.show(); print("Entrenamiento terminado. Mejor eval:", mejor)


## 9. Evaluacion final

Cargamos el mejor modelo y corremos varios episodios greedy (politica aprendida, sin
exploracion). Reportamos kills y supervivencia: las metricas que importan en la sala.

In [ ]:
best_agent = DQNAgent(env.state_dim, env.n_actions,
                      device="cuda" if torch.cuda.is_available() else "cpu")
best_agent.load(CKPT)
ev_env = make_env()
for e in range(5):
    s = ev_env.reset(); info = {}; steps = 0
    for steps in range(1, MAX_STEPS+1):
        s,r,d,info = ev_env.step(best_agent.act(s, greedy=True, forbidden=forbidden))
        if d: break
    print(f"episodio {e+1}: kills {int(info.get('kills',0))} | pasos vivo {steps}")
ev_env.close()


## 10. Conclusiones

- La **percepcion YOLO** (entrenada con auto-etiquetado desde ViZDoom) entrega un estado
  semantico e interpretable, mas eficiente en muestras que aprender desde pixeles crudos.
- El agente **Double DQN** aprende a defender la sala como torreta: gira para barrer, encara
  y dispara a los enemigos, conservando municion.
- Trabajo futuro: mas timesteps, mas clases de enemigos en el dataset, y comparar contra un
  baseline de DQN sobre pixeles.